In [17]:
import numpy as np
import ee
import json
from datetime import datetime
import os
import time
from tqdm import tqdm
from google.colab import files
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Danh sách tỉnh ĐBSCL
MEKONG_PROVINCES = [
    'An Giang', 'Ben Tre', 'Ca Mau', 'Can Tho city', 'Dong Thap',
    'Hau Giang', 'Kien Giang', 'Long An', 'Soc Trang',
    'Tien Giang', 'Tra Vinh', 'Vinh Long', 'Bac Lieu'
]

# Khởi tạo Earth Engine
try:
    ee.Initialize(project='ee-python-api-471906')
    print("Earth Engine initialized successfully")
except Exception as e:
    print(f"Failed to initialize Earth Engine: {e}")
    ee.Authenticate()
    ee.Initialize(project='ee-python-api-471906')

def get_mekong_region():
    """Lấy geometry của vùng ĐBSCL"""
    try:
        provinces = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam'))
        mekong_fc = provinces.filter(ee.Filter.inList('ADM1_NAME', MEKONG_PROVINCES))
        return mekong_fc.union().geometry()
    except Exception as e:
        print(f"Error getting Mekong region: {e}")
        raise

def add_derived_features(image):
    """Thêm tất cả derived features từ VV và VH"""
    vv = image.select('VV')
    vh = image.select('VH')
    angle = image.select('angle')

    # Tính tất cả features cùng lúc
    ratio = vv.divide(vh).rename('VV_VH_ratio')
    nd = vv.subtract(vh).divide(vv.add(vh)).rename('ND_VV_VH')
    cross_ratio = vh.divide(vv).rename('VH_VV_ratio')
    rvi = vh.multiply(4).divide(vv.add(vh)).rename('RVI')
    dpsvi = vv.add(vh).divide(2).rename('DPSVI')
    total = vv.add(vh).rename('Total_Backscatter')
    diff = vv.subtract(vh).rename('Diff_Backscatter')
    prod = vv.multiply(vh).rename('Prod_Backscatter')

    return image.addBands([ratio, nd, cross_ratio, rvi, dpsvi, total, diff, prod, angle])

def get_s1_collection(region, start_date, end_date):
    """Lấy bộ sưu tập dữ liệu Sentinel-1"""
    try:
        collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                      .filterBounds(region)
                      .filterDate(start_date, end_date)
                      .filter(ee.Filter.eq('instrumentMode', 'IW'))
                      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                      .filter(ee.Filter.eq('resolution_meters', 10))
                      .sort('system:time_start'))

        size = collection.size().getInfo()
        if size == 0:
            print(f"No Sentinel-1 images found for {start_date} to {end_date}")
            return None
        print(f"Found {size} Sentinel-1 images")
        return collection
    except Exception as e:
        print(f"Error getting Sentinel-1 collection: {e}")
        return None

def compute_all_features_fast(image, region):
    """
    Tính toán TẤT CẢ features statistics trong MỘT lần gọi API duy nhất
    Tối ưu tốc độ bằng cách dùng reducer kết hợp
    """
    try:
        image_with_features = add_derived_features(image)

        all_bands = ['VV', 'VH', 'angle', 'VV_VH_ratio', 'ND_VV_VH',
                     'VH_VV_ratio', 'RVI', 'DPSVI', 'Total_Backscatter',
                     'Diff_Backscatter', 'Prod_Backscatter']

        # Tạo reducer kết hợp cho TẤT CẢ thống kê cùng lúc
        combined_reducer = (
            ee.Reducer.mean()
            .combine(ee.Reducer.stdDev(), sharedInputs=True)
            .combine(ee.Reducer.min(), sharedInputs=True)
            .combine(ee.Reducer.max(), sharedInputs=True)
            .combine(ee.Reducer.percentile([25, 50, 75]), sharedInputs=True)
        )

        # Tính TẤT CẢ băng tần trong MỘT lần gọi duy nhất
        stats = image_with_features.select(all_bands).reduceRegion(
            reducer=combined_reducer,
            geometry=region,
            scale=200,  # Tăng scale để tính nhanh hơn
            maxPixels=5e8,  # Giảm maxPixels
            bestEffort=True,  # Quan trọng: cho phép tính nhanh hơn
            tileScale=4  # Tăng tileScale để giảm memory
        ).getInfo()

        return stats
    except Exception as e:
        print(f"  ⚠ Error computing features: {e}")
        return None

def save_metadata_fast(image, region, output_subdir):
    """Lưu metadata với tất cả features - tối ưu tốc độ"""
    try:
        image_id = image.get('system:index').getInfo()

        # Lấy properties cơ bản
        properties = {
            'system:index': image_id,
            'system:time_start': image.get('system:time_start').getInfo(),
            'orbitProperties_pass': image.get('orbitProperties_pass').getInfo(),
            'platform_number': image.get('platform_number').getInfo(),
            'instrumentMode': image.get('instrumentMode').getInfo(),
            'resolution_meters': image.get('resolution_meters').getInfo(),
        }

        # Thêm thông tin features
        properties['derived_features'] = [
            'VV_VH_ratio', 'ND_VV_VH', 'VH_VV_ratio',
            'RVI', 'DPSVI', 'Total_Backscatter',
            'Diff_Backscatter', 'Prod_Backscatter'
        ]
        properties['original_bands'] = ['VV', 'VH', 'angle']
        properties['feature_descriptions'] = {
            'VV': 'VV polarization backscatter (dB)',
            'VH': 'VH polarization backscatter (dB)',
            'angle': 'Incidence angle',
            'VV_VH_ratio': 'VV/VH cross-polarization ratio',
            'ND_VV_VH': 'Normalized Difference (VV-VH)/(VV+VH)',
            'VH_VV_ratio': 'VH/VV ratio',
            'RVI': 'Radar Vegetation Index = 4*VH/(VV+VH)',
            'DPSVI': 'Dual-pol SAR Vegetation Index = (VV+VH)/2',
            'Total_Backscatter': 'VV + VH',
            'Diff_Backscatter': 'VV - VH',
            'Prod_Backscatter': 'VV * VH'
        }

        # Tính toán features statistics - CHỈ MỘT lần gọi API
        features_stats = compute_all_features_fast(image, region)
        if features_stats:
            properties['features_statistics'] = features_stats

        # Lưu file JSON
        metadata_filename = f"{image_id.replace('/', '_')}_metadata.json"
        metadata_path = os.path.join(output_subdir, metadata_filename)
        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(properties, f, ensure_ascii=False, indent=2)

        return metadata_path
    except Exception as e:
        print(f"  ⚠ Error saving metadata: {e}")
        return None

def process_single_image(img_info, region, output_dir):
    """Xử lý một ảnh đơn lẻ - dùng cho parallel processing"""
    try:
        img, idx = img_info
        image_date_millis = img.get('system:time_start').getInfo()
        date_time_str = datetime.fromtimestamp(image_date_millis / 1000).strftime('%Y-%m-%d_%H-%M-%S')
        image_subdir = os.path.join(output_dir, f"S1_{date_time_str}")
        os.makedirs(image_subdir, exist_ok=True)

        # Chỉ lưu metadata
        metadata_path = save_metadata_fast(img, region, image_subdir)

        if metadata_path:
            return {
                'success': True,
                'date': date_time_str,
                'metadata': metadata_path,
                'idx': idx
            }
        else:
            return {'success': False, 'idx': idx}

    except Exception as e:
        return {'success': False, 'idx': idx, 'error': str(e)}

def process_batch_images_parallel(collection, region, output_dir, max_images=None, max_workers=8):
    """
    Xử lý hàng loạt ảnh Sentinel-1 SONG SONG với ThreadPoolExecutor
    max_workers: số luồng chạy đồng thời (mặc định 8)
    """
    if max_images == None:
        images_list = collection.toList(collection.size())
        total_images = images_list.size().getInfo()
    else:
        images_list = collection.toList(max_images)
        total_images = min(max_images, images_list.size().getInfo())

    # Chuẩn bị danh sách các ảnh với index
    print(f"🔄 Preparing {total_images} images for parallel processing...")
    images_to_process = []
    for i in range(total_images):
        img = ee.Image(images_list.get(i)).clip(region)
        images_to_process.append((img, i))

    results = []
    success_count = 0
    error_count = 0

    print(f"🚀 Processing with {max_workers} parallel workers...")

    # Sử dụng ThreadPoolExecutor để xử lý song song
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit tất cả tasks
        future_to_img = {
            executor.submit(process_single_image, img_info, region, output_dir): img_info[1]
            for img_info in images_to_process
        }

        # Progress bar
        with tqdm(total=total_images, desc="Processing", unit="img") as pbar:
            for future in as_completed(future_to_img):
                result = future.result()
                if result['success']:
                    results.append(result)
                    success_count += 1
                    pbar.set_postfix_str(f"✓ {success_count}/{success_count+error_count}")
                else:
                    error_count += 1
                    if 'error' in result:
                        pbar.write(f"❌ Image {result['idx']}: {result['error'][:80]}")
                pbar.update(1)

    return results

def main():
    year = 2024
    months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

    # Cấu hình số workers (luồng song song)
    MAX_WORKERS = 16  # Tăng lên 10 để xử lý nhanh hơn (có thể điều chỉnh 4-16)

    print("🌍 Getting Mekong Delta region...")
    mekong_region = get_mekong_region()
    print("✅ Region loaded\n")

    total_processed = 0

    for idx in range(len(months)):
        OUTPUT_DIR = f"/content/dbscl-sentinel-1_{year}-{months[idx]}"
        os.makedirs(OUTPUT_DIR, exist_ok=True)

        start_date = f'{year}-{months[idx]}-01'
        if idx == len(months) - 1:
            end_date = f'{year}-12-31'
        else:
            end_date = f'{year}-{months[idx+1]}-01'

        print(f"\n📅 Month {months[idx]}/{year} ({start_date} to {end_date})")
        collection = get_s1_collection(mekong_region, start_date, end_date)

        if collection is None:
            print(f"⏭️  No images found, skipping...\n")
            continue

        # Xử lý SONG SONG với multi-threading
        results = process_batch_images_parallel(
            collection,
            mekong_region,
            OUTPUT_DIR,
            max_workers=MAX_WORKERS
        )
        total_processed += len(results)
        print(f"✅ Completed: {len(results)} images\n")

    print(f"\n🎉 DONE! Total processed: {total_processed} images")
    print("📦 Creating zip file...")

    all_months_zip = f'/content/dbscl-sentinel-1_{year}_metadata.zip'
    os.system(f'zip -r -q {all_months_zip} /content/dbscl-sentinel-1_{year}*')

    print("⬇️  Downloading...")
    files.download(all_months_zip)
    print("✅ Complete!")

if __name__ == "__main__":
    main()

Earth Engine initialized successfully
🌍 Getting Mekong Delta region...
✅ Region loaded


📅 Month 01/2024 (2024-01-01 to 2024-02-01)
Found 25 Sentinel-1 images
🔄 Preparing 25 images for parallel processing...
🚀 Processing with 16 parallel workers...


Processing: 100%|██████████| 25/25 [00:25<00:00,  1.01s/img, ✓ 25/25]


✅ Completed: 25 images


📅 Month 02/2024 (2024-02-01 to 2024-03-01)
Found 25 Sentinel-1 images
🔄 Preparing 25 images for parallel processing...
🚀 Processing with 16 parallel workers...


Processing: 100%|██████████| 25/25 [00:26<00:00,  1.06s/img, ✓ 25/25]


✅ Completed: 25 images


📅 Month 03/2024 (2024-03-01 to 2024-04-01)
Found 25 Sentinel-1 images
🔄 Preparing 25 images for parallel processing...
🚀 Processing with 16 parallel workers...


...

Processing: 100%|██████████| 24/24 [00:23<00:00,  1.03img/s, ✓ 24/24]

✅ Completed: 24 images


🎉 DONE! Total processed: 288 images
📦 Creating zip file...
⬇️  Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Complete!
